# PD-001 — bounded performance pilot

Run All in the existing Kaggle Notebook with Internet and one T4. Default: M0 and P50-F16, B0 versus LIGHT only, 1 warmup + 5 measured steps, one repeat: **4 rounds**, not 42. These short measurements are pilot evidence, not a conclusive statistical benchmark.

The entire launch (checkout, measurement and checkpoint rehearsal) has a **120-minute wall limit**. Measurement itself has a 30-minute cap. Completed measurements are saved after each round. At the limit, all launcher/worker processes are stopped; Save Version to retain Output. No full training or confirmation data are authorized. The ordinary training launcher pauses at a completed step with 10 minutes reserved for checkpoint/export, and shares the same hard limit.


In [ ]:
"""Standard-library session supervisor, also embedded verbatim before Notebook checkout."""
import json
import os
from pathlib import Path
import signal
import subprocess
import time

SESSION_SECONDS = 7200
SAVE_RESERVE_SECONDS = 600
TERMINATION_RESERVE_SECONDS = 10
DEADLINE_ENV = "PD001_SESSION_DEADLINE_MONOTONIC"


def session_deadline():
    now = time.monotonic()
    requested = float(os.environ.get(DEADLINE_ENV, now + SESSION_SECONDS))
    if not now < requested < float("inf"):
        raise TimeoutError("PD-001 session budget exhausted; preserve Output")
    deadline = min(requested, now + SESSION_SECONDS)
    os.environ[DEADLINE_ENV] = str(deadline)
    return deadline


def pause_due():
    value = os.environ.get(DEADLINE_ENV)
    return value is not None and time.monotonic() >= float(value) - SAVE_RESERVE_SECONDS


class SessionBudget:
    def __init__(self, status_path):
        self.deadline = session_deadline()
        self.status_path = Path(status_path)

    def record(self, status):
        self.status_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = self.status_path.with_suffix(".tmp")
        temporary.write_text(json.dumps({
            "status": status, "wall_limit_seconds": SESSION_SECONDS,
            "remaining_seconds": max(0, self.deadline - time.monotonic()),
            "scientific_decision": False,
            "note": "Retain Kaggle Output; only completed checkpoints/evidence can be restored."
        }, indent=2) + "\n")
        temporary.replace(self.status_path)

    def run(self, command, **kwargs):
        remaining = self.deadline - time.monotonic() - TERMINATION_RESERVE_SECONDS
        if remaining <= 0:
            self.record("SESSION_TIME_LIMIT")
            raise TimeoutError("PD-001 two-hour session limit; preserve Output")
        # One group contains the launcher, benchmark workers and training workers.
        # Killing only the immediate launcher can leave CUDA work running.
        process = subprocess.Popen(command, start_new_session=True, **kwargs)
        try:
            code = process.wait(timeout=remaining)
        except BaseException as error:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                pass
            # Group members may outlive an already exited leader.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=3)
            self.record("SESSION_TIME_LIMIT" if isinstance(error, subprocess.TimeoutExpired) else "INTERRUPTED")
            raise
        if code:
            self.record("FAILED")
            raise subprocess.CalledProcessError(code, command)
        return code

BUDGET = SessionBudget('/kaggle/working/pd001-session-status.json')
BUDGET.record('RUNNING')
print('Session wall limit: 120 minutes; training checkpoint pause begins with 10 minutes left',flush=True)

from pathlib import Path
import subprocess, sys, os, json
SOURCE_COMMIT = '4927956e529d3401364fa2b9bce70a6c62831c73'
BASELINE_COMMIT = '2e77369b0df1ce704863a815dacc48301736dde0'
WARMUP, STEPS, REPEATS = 1, 5, 1
RESTORE_FROM = ""  # optional: portable directory from a RETAINED previous Kaggle Output
CASES = ['B0','LIGHT']
WORK = Path('/kaggle/working/pd001-performance-' + SOURCE_COMMIT[:12])
REL = 'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # both arms measured sequentially on the same GPU
if SOURCE_COMMIT.startswith('__'):
    raise RuntimeError('Import the committed performance notebook with its pinned source')
WORK.mkdir(parents=True,exist_ok=True)
def checkout(name, commit):
    root = WORK/name
    if (root/'.git').exists():
        head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=root,text=True).strip()
        if head != commit:
            raise RuntimeError('Existing code checkout identity mismatch; preserve outputs')
        return root
    root.mkdir(parents=True,exist_ok=False)
    def git(*args):
        print('[Setup] ' + ' '.join(args),flush=True)
        BUDGET.run(['git',*args],cwd=root)
    git('init','-b','performance')
    git('remote','add','origin','https://github.com/ArcheLabs/mini-cells.git')
    git('sparse-checkout','init','--no-cone')
    git('sparse-checkout','set','--no-cone','/pyproject.toml','/src/','/shared/python/',
        '/research/projects/cellular-language-models/experiments/006-path-first-clm/',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/')
    git('fetch','--depth=1','--filter=blob:none','origin',commit)
    git('checkout','--detach','FETCH_HEAD')
    return root
ROOT = checkout('repaired-code', SOURCE_COMMIT)
BASELINE = checkout('baseline-code', BASELINE_COMMIT)

ENTRY = ROOT/REL/'run_pd001.py'


In [ ]:
if not RESTORE_FROM:
    ENTRY = ROOT/REL/'run_pd001.py'
    OUTPUT = WORK/'measurements'
    if OUTPUT.exists():
        raise RuntimeError('Performance evidence already exists; download it or start a fresh output folder')
    command = [sys.executable,str(ENTRY),'--mode','perf-audit','--all-arms',
               '--baseline-root',str(BASELINE),'--device','cuda:0',
               '--allow-gpu-microbenchmark','--warmup',str(WARMUP),'--steps',str(STEPS),
               '--repeats',str(REPEATS),'--cases',*CASES,'--output',str(OUTPUT)]
    print('Sequential synthetic microbenchmarks; no full training',flush=True)
    BUDGET.run(command,cwd=ROOT)
    report = json.loads((OUTPUT/'ablation.json').read_text())
    print(json.dumps(report['summary'],indent=2))
    print('Download/Save Version output:',OUTPUT)
    print('READY_FOR_RETRAIN = NO; benchmark review and actual Kaggle persistence rehearsal remain required')


## Cross-session checkpoint rehearsal

After the benchmark, this notebook creates a bounded **two-step synthetic** P50-F16 checkpoint. Save Version to retain it. In a new cloud session attach that Notebook Output as input and set `RESTORE_FROM` to the `portable` directory shown below. Run All will skip the benchmark, restore the checkpoint, perform the remaining two steps and compare against a fresh uninterrupted four-step control. This is still engineering validation, not a language experiment. A local process restart is not evidence that Kaggle retained the files; record both Kaggle version links when reviewing the gate.


In [ ]:
PHASE = 'resume' if RESTORE_FROM else 'pause'
REHEARSAL_OUTPUT = WORK/('resume-rehearsal-' + PHASE)
cmd = [sys.executable,str(ENTRY),'--mode','resume-audit','--arm','P50-F16',
       '--source-commit',SOURCE_COMMIT,'--device','cuda:0','--allow-gpu-microbenchmark',
       '--rehearsal-phase',PHASE,'--output',str(REHEARSAL_OUTPUT)]
if RESTORE_FROM:
    cmd += ['--restore-from',RESTORE_FROM]
BUDGET.run(cmd,cwd=ROOT)
print((REHEARSAL_OUTPUT/'resume-audit.json').read_text())
if PHASE == 'pause':
    print('Save Version must retain this Output. In a NEW cloud session, attach that Output, set RESTORE_FROM to its portable directory, and Run All again. This verifies persistence rather than just reusing /working.')

BUDGET.record('COMPLETED')
